# Day 3 - Visual tokens

- status: implemented_toy_not_executed_real_model
- stage: VLM_DAY_3
- paper_ids: qwen2_vl_2024, qwen3_vl_2025
- dataset_ids: synthetic_toy, user_selected_nas_data
- seed: 42
- scope: educational implementation; real inference/training is opt-in

이 노트북은 다른 사용자 파일, 공유 환경, checkpoint를 자동으로 변경하지 않는다. 실제 데이터는
`/nas/datahub/min` 아래 사용자가 지정한 경로만 읽는다.

## 1. Learning question

dynamic resolution에서 한 이미지가 몇 visual token을 차지하며, 해상도 선택이 context·memory·작은 물체에 어떤 trade-off를 만드는가?

## 2. Background theory

Qwen3-VL은 16x16 patch와 2x2 spatial merge를 사용한다. 정렬된 이미지에서 LLM visual token 하나는 대략 32x32=1024 pixel 영역에 대응한다. 실제 processor는 min/max pixels와 aspect ratio를 고려해 smart resize한다.

## 3. Paper connection

Qwen-VL 계열의 dynamic resolution은 원본 aspect ratio와 detail을 더 유연하게 유지한다. Qwen3-VL은 Interleaved-MRoPE로 time/height/width position을 다루고, DeepStack으로 여러 ViT layer feature를 LLM에 연결한다.

## 4. Input/output and shapes

이미지 `[H,W]`는 32의 배수로 맞춰지고 pre-merge grid `[H/16,W/16]`, LLM visual token은 `(H/16)*(W/16)/4`다. 전체 sequence는 text + image/video token이다.

In [ ]:
from pathlib import Path
import sys
import numpy as np

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "pyproject.toml").is_file()),
    Path("/nas/home/mhlee/vlm-foundation-7days"),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project:", PROJECT_ROOT)
print("numpy:", np.__version__)

## 5. Minimal implementation

여러 해상도의 token 수와 attention cost proxy를 계산한다. 이 함수는 token accounting용이며 공식 smart_resize의 대체 구현이 아니다.

In [ ]:
from vlm_foundation.visual_tokens import self_attention_elements, visual_token_budget

for height, width in [(224, 224), (640, 960), (1080, 1920), (2160, 3840)]:
    budget = visual_token_budget(height, width)
    print((height, width), budget)

## 6. Visualization sanity check

resized H/W가 32의 배수인지, 극단적 aspect ratio에서 token이 폭증하지 않는지 표로 확인한다.

In [ ]:
budgets = [visual_token_budget(h, w) for h, w in [(224,224), (640,960), (1080,1920)]]
assert all(b.resized_height % 32 == 0 and b.resized_width % 32 == 0 for b in budgets)
print("alignment check: PASS")

## 7. Experiment

같은 text prompt에 224, 640x960, 4K 이미지를 넣을 때 sequence-squared 비용을 비교한다.

In [ ]:
text_tokens = 512
for height, width in [(224, 224), (640, 960), (2160, 3840)]:
    visual = visual_token_budget(height, width).llm_visual_tokens
    print((height, width), "visual=", visual, "sequence^2=", self_attention_elements(text_tokens, visual))

## 8. Metrics

visual token 수, preprocess 시간, peak memory, time-to-first-token, grounding/OCR accuracy를 함께 본다.

## 9. Interpretation

해상도는 품질 knob이면서 비용 knob이다. max_pixels를 올리면 무조건 좋아지는 것이 아니라 context와 batch capacity를 소모한다.

## 10. Failure cases

과도한 downscale의 작은 글자 누락, 과도한 upscale, panorama token 폭증, video frame 수와 pixel budget 곱셈을 확인한다.

## 11. Real-service implications

업무별 resolution policy와 total visual-token budget을 둔다. 요청자가 임의 4K multi-image를 보내 전체 queue를 막지 않도록 제한한다.

## 12. Review questions

1. 640x960은 몇 LLM visual token인가?
2. 2x2 merge 전후 token 수 차이는?
3. image token 증가가 text context에 주는 영향은?
4. OCR과 captioning의 해상도 정책이 다른 이유는?
5. video에서는 어떤 두 축이 token을 늘리는가?